# Tutorial: reading the Schmidt Observatory System S3 bucket

The survey products live in the private S3 bucket `schmidt-observatory-system`. On the hub the bucket is
also NFS-mounted read-only at `~/s3/schmidt-observatory-system`, so every object is a plain file. This notebook shows both routes on one DSA
mock FITS image, then opens the Argus visit store from S3.

**Credentials.** Anonymous access is denied. boto3 and pyarrow both use the default AWS credential chain
(the hub's role, `~/.aws/credentials`, or `AWS_ACCESS_KEY_ID`/`AWS_SECRET_ACCESS_KEY`). Nothing in this
notebook hard-codes a key.

In [ ]:
BUCKET = "schmidt-observatory-system"
SAMPLE_FILE = "dsa/v2-convening/mockfm_DPR-DSA-03_mp37_701.235_20260922.fits"
MOUNT = "~/s3/schmidt-observatory-system"   # the bucket, NFS-mounted read-only on the hub
CUTOUT = 256        # cutout size in pixels, per side
STORAGE_OPTIONS = {}  # extra pyarrow.fs.S3FileSystem kwargs, e.g. {"region": "us-east-1"}

import io
import os
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.fs as pafs
from astropy.io import fits


## Step 1. List and inspect objects with boto3

`list_objects_v2` is paginated at 1000 keys; the paginator hides that. `head_object` returns size and
metadata without downloading.

In [ ]:
s3 = boto3.client("s3")
prefix = SAMPLE_FILE.rsplit("/", 1)[0] + "/"
keys = [o for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix)
        for o in page.get("Contents", [])]
print(f"{len(keys)} objects under s3://{BUCKET}/{prefix}")
for o in keys[:10]:
    print(f"  {o['Size'] / 1e6:8.1f} MB  {o['Key']}")

head = s3.head_object(Bucket=BUCKET, Key=SAMPLE_FILE)
print(f"\n{SAMPLE_FILE}: {head['ContentLength'] / 1e6:.1f} MB, last modified {head['LastModified']:%Y-%m-%d}")

The same listing from a shell: `aws s3 ls s3://schmidt-observatory-system/dsa/v2-convening/`, and
`aws s3 cp s3://schmidt-observatory-system/<key> .` to download.

## Step 2. Read the FITS file from S3

`get_object` streams the bytes; `astropy.io.fits` opens them from memory, so nothing is written to disk.
For a large file, `s3.download_file(BUCKET, SAMPLE_FILE, local_path)` is the alternative.

In [ ]:
data = s3.get_object(Bucket=BUCKET, Key=SAMPLE_FILE)["Body"].read()
print(f"read {len(data) / 1e6:.1f} MB")
hdul = fits.open(io.BytesIO(data))
hdul.info()

In [ ]:
# first HDU with image data, squeezed to 2-D (radio cubes often carry degenerate axes)
hdu = next(h for h in hdul if h.data is not None and h.data.ndim >= 2)
img = np.squeeze(hdu.data).astype(float)
while img.ndim > 2:
    img = img[0]
print(f"image {img.shape}, finite pixels {np.isfinite(img).mean():.1%}, median {np.nanmedian(img):.4g}")
for k in ("OBJECT", "DATE-OBS", "BUNIT", "CTYPE1", "CTYPE2", "CRVAL1", "CRVAL2"):
    if k in hdu.header:
        print(f"  {k:8s} = {hdu.header[k]}")

## Step 3. A central cutout

In [ ]:
cy, cx = img.shape[0] // 2, img.shape[1] // 2
h = CUTOUT // 2
cut = img[max(cy - h, 0):cy + h, max(cx - h, 0):cx + h]
lo, hi = np.nanpercentile(cut, [1, 99.5])
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cut, origin="lower", cmap="gray", vmin=lo, vmax=hi)
fig.colorbar(im, ax=ax, label=hdu.header.get("BUNIT", "pixel value"))
ax.set_title(f"{CUTOUT} px cutout of {Path(SAMPLE_FILE).name}", fontsize=9)
plt.show()
print(f"cutout {cut.shape}, 1-99.5% range {lo:.4g} to {hi:.4g}")

## Step 4. The same file through the NFS mount

On the hub, `~/s3/<key>` is the object. This needs no credentials and lets `fits.open` read lazily
(`memmap`), which is the better route for a few large files. The cell is skipped where the mount is absent.

In [ ]:
local = Path(MOUNT).expanduser() / SAMPLE_FILE
if local.is_file():
    with fits.open(local) as h2:
        same = np.array_equal(np.squeeze(h2[hdul.index(hdu)].data), np.squeeze(hdu.data))
    print(f"{local}: {local.stat().st_size / 1e6:.1f} MB, identical to the S3 read: {same}")
else:
    print(f"{local} not found: the bucket is not mounted here, S3 is the only route.")

## Step 5. The Argus visit store from S3

`argus_sim.lightcurve.SurveyRun` reads a store through `pyarrow.fs.S3FileSystem`; each query is one or two
ranged GETs. `storage_options` are the `S3FileSystem` keywords (region, endpoint_override, access_key, ...).
We list the runs in the bucket, open v5.4 and count the reads one position query costs.

In [ ]:
fs = pafs.S3FileSystem(**STORAGE_OPTIONS)
runs = fs.get_file_info(pafs.FileSelector(f"{BUCKET}/argus/sim/", recursive=False))
print("survey runs in the bucket:")
for r in runs:
    print("  s3://" + r.path)

In [ ]:
from argus_sim.lightcurve import SurveyRun

STORE = os.path.expanduser(f"{MOUNT}/argus/sim/argsim_5yr_beta_rho_uniform_v5.4/store")
run = SurveyRun(STORE)   # the mount: plain file reads, no credentials
print(run)
before = run.io_stats()
vis = run.visits(223.878, 36.031, band="g", mjd_min=62200, mjd_max=62230)
after = run.io_stats()
print(f"{len(vis):,} g slots in 30 nights; the query made {after['get_requests'] - before['get_requests']} GET requests for {(after['bytes_read'] - before['bytes_read']) / 1e6:.1f} MB")

## Summary

- `boto3.client("s3")` lists, inspects and streams objects; `aws s3 ls` / `aws s3 cp` do the same from a shell.
- `fits.open(io.BytesIO(bytes))` reads a FITS file straight from S3; `~/s3/<key>` is the same file on the hub.
- `SurveyRun("s3://...", storage_options={...})` reads the Argus visit store in place, a few GETs per query.

Continue with `tutorial_survey_data.ipynb`, whose first cell picks the mount or S3 automatically.